# IRFZ44N MOSFET Analytical Loss Calculation

This notebook estimates the conduction loss, switching loss and total
average power loss of the IRFZ44N MOSFET used in the buck converter.

The analytical result will later be compared with the average MOSFET
power obtained from LTspice.

In [1]:
# Buck converter operating conditions

Vin = 24.0          # Input voltage, V
Vout = 12.0         # Output voltage, V
Iout = 10.0         # Output current, A
fsw = 50_000        # Switching frequency, Hz
Vgs = 10.0          # Gate-source drive voltage, V

D = Vout / Vin      # Ideal buck-converter duty ratio

print(f"Duty ratio = {D:.3f}")

Duty ratio = 0.500


## MOSFET parameters

The on-state resistance used in this calculation is 0.0139 Ω, matching
the value used by the IRFZ44N LTspice model. This allows a more consistent
comparison between the analytical calculation and the LTspice average
power-loss result.

The rise and fall times are taken from the manufacturer datasheet and
are used to estimate switching loss.

In [2]:
# IRFZ44N MOSFET parameters

Rds_on = 0.0139      # LTspice model on-resistance, ohms
tr = 60e-9           # Datasheet rise time, seconds
tf = 45e-9           # Datasheet fall time, seconds

In [3]:
import math

I_mosfet_rms = Iout * math.sqrt(D)

P_cond = I_mosfet_rms**2 * Rds_on

print(f"MOSFET RMS current = {I_mosfet_rms:.3f} A")
print(f"Conduction loss = {P_cond:.3f} W")

MOSFET RMS current = 7.071 A
Conduction loss = 0.695 W


In [4]:
P_sw = 0.5 * Vin * Iout * (tr + tf) * fsw

print(f"Switching loss = {P_sw:.3f} W")

Switching loss = 0.630 W


In [5]:
P_total = P_cond + P_sw

print(f"Conduction loss = {P_cond:.3f} W")
print(f"Switching loss  = {P_sw:.3f} W")
print(f"Total MOSFET loss = {P_total:.3f} W")

Conduction loss = 0.695 W
Switching loss  = 0.630 W
Total MOSFET loss = 1.325 W


In [6]:
import pandas as pd

results = pd.DataFrame({
    "Loss component": [
        "Conduction loss",
        "Switching loss",
        "Total MOSFET loss"
    ],
    "Power loss (W)": [
        P_cond,
        P_sw,
        P_total
    ]
})

results

,Loss component,Power loss (W)
0,Conduction loss,0.695
1,Switching loss,0.630
2,Total MOSFET loss,1.325


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv(
    "BUCK converter",
    sep=r"\s+",
    engine="python"
)

print(data.head())
print(data.columns)

           time    V(Vin,Vsw)        Id(M1)
0  0.000000e+00  0.000000e+00  1.253150e-09
1  2.000000e-14  5.417492e-09  1.856440e-06
2  4.000000e-14  1.088395e-08  3.706178e-06
3  8.000000e-14  2.191459e-08  7.392287e-06
4  1.600000e-13  4.455971e-08  1.468470e-05
Index(['time', 'V(Vin,Vsw)', 'Id(M1)'], dtype='str')


In [12]:
# Convert all imported columns to numbers
data["time"] = pd.to_numeric(data["time"], errors="coerce")
data["V(Vin,Vsw)"] = pd.to_numeric(data["V(Vin,Vsw)"], errors="coerce")
data["Id(M1)"] = pd.to_numeric(data["Id(M1)"], errors="coerce")

# Remove any invalid or empty rows
data = data.dropna(
    subset=["time", "V(Vin,Vsw)", "Id(M1)"]
).copy()

# Keep only the steady-state section from 4 ms to 5 ms
steady_state = data[
    (data["time"] >= 0.004) &
    (data["time"] <= 0.005)
].copy()

print(steady_state.head())
print("Number of points:", len(steady_state))

Empty DataFrame
Columns: [time, V(Vin,Vsw), Id(M1)]
Index: []
Number of points: 0


In [13]:
time = steady_state["time"].to_numpy()
vds = steady_state["V(Vin,Vsw)"].to_numpy()
ids = steady_state["Id(M1)"].to_numpy()

# Instantaneous MOSFET power
power = vds * ids

# Time-averaged MOSFET power
average_power = np.trapezoid(power, time) / (
    time[-1] - time[0]
)

print(f"Average MOSFET power from exported data = {average_power:.4f} W")

IndexError: index -1 is out of bounds for axis 0 with size 0

## Conclusion

The calculated analytical MOSFET power loss is higher than the LTspice result because it is based on simplified and conservative datasheet assumptions. The switching-loss equation assumes voltage and current overlap throughout the full rise and fall times, which can overestimate the actual switching loss.

LTspice instead calculates power from the simulated drain voltage and current waveforms. In this model, the ideal gate-voltage source causes very fast switching, reducing the effective voltage-current overlap and therefore reducing switching loss.

For this reason, the LTspice average power loss of **0.742 W is reasonable for the simulated circuit**, while the analytical result of approximately **1.3–1.5 W should be treated as a more conservative engineering estimate**.